Challenge 3

# Challenge 3

## Step 1: Set up

### Install libs

In [1]:
!pip install google-genai "google-cloud-aiplatform[evaluation]" ipytest

### Imports

In [2]:
import os
import datetime
import pandas as pd

from google import genai
from google.genai import types

import vertexai
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

import ipytest
ipytest.autoconfig()

### Config

In [3]:
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "qwiklabs-gcp-02-980c840fda76")
LOCATION = "us-east1"
GEMINI_MODEL = "gemini-2.5-flash"

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
vertexai.init(project=PROJECT_ID, location=LOCATION)

## Step 2: Define generative AI functions

### Classifier function

In [4]:
def classify_question(question):
    """Classify a citizen question into one of four town-service categories."""
    system = (
        "You classify citizen questions for a town government assistant. "
        "Respond with exactly one of these labels and nothing else: "
        "Employment, General Information, Emergency Services, Tax Related."
    )
    resp = genai_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=question,
        config=types.GenerateContentConfig(system_instruction=system, temperature=0),
    )
    return resp.text.strip()

### Announcement generator function

In [5]:
def generate_announcement_post(topic):
    """Generate a short official social media post for a government announcement."""
    system = (
        "You write official social media posts for the town of Aurora Bay.\n"
        "Rules:\n"
        "1. Keep the post under 280 characters.\n"
        "2. Use a clear, professional public-service tone.\n"
        "3. End with the hashtag #AuroraBay."
    )
    resp = genai_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=f"Write a post announcing: {topic}",
        config=types.GenerateContentConfig(system_instruction=system),
    )
    return resp.text.strip()

## Step 3: Run unit tests

### Classifier unit test

In [6]:
%%ipytest

def test_classifier():
    assert classify_question("How do I apply for a job with the city?") == "Employment"
    assert classify_question("There is a gas leak on Main Street, who do I call?") == "Emergency Services"
    assert classify_question("When are my property taxes due?") == "Tax Related"
    assert classify_question("What are the public library hours?") == "General Information"

.                                                                                            [100%]
========================================= warnings summary =========================================
../usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290
  /usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 1 warning in 5.53s


### Announcement generator

In [9]:
%%ipytest

def test_announcement_generator():
    rules = (
        "Write an official town government social media post announcing: {topic}. "
        "It must be under 280 characters, use a clear professional public-service "
        "tone, and end with the hashtag #AuroraBay."
    )
    topics = [
        "the town pool opens for summer on June 1",
        "a snow emergency tonight; residents should avoid all roads",
        "city hall is closed for the Thanksgiving holiday",
    ]
    df = pd.DataFrame([
        {
            "prompt": rules.format(topic=t),
            "instruction": rules.format(topic=t),
            "response": generate_announcement_post(t),
        }
        for t in topics
    ])
    result = EvalTask(
        dataset=df,
        metrics=[
            MetricPromptTemplateExamples.Pointwise.INSTRUCTION_FOLLOWING,
            MetricPromptTemplateExamples.Pointwise.COHERENCE,
            MetricPromptTemplateExamples.Pointwise.SAFETY,
        ],
        experiment="announcement-eval",
    ).evaluate()

    assert result.summary_metrics["instruction_following/mean"] >= 4.0
    assert result.summary_metrics["coherence/mean"] >= 4.0
    assert result.summary_metrics["safety/mean"] == 1.0

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 9 Vertex Gen AI Evaluation Service API requests.
INFO:vertexai.evaluation._evaluation:All 9 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:18.88586324100106 seconds


.                                                                                            [100%]
========================================= warnings summary =========================================
../usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290
  /usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 1 warning in 49.05s
